# Multi-format model extension benchmark (credit application)

German credit deployment CSV: **50 rows**, **48** schema features, serial `id`, empty `class`.

This notebook:

1. Builds canonical `input_with_empty_column.csv` from the credit inference source
2. Runs `create_model.py` in every extension folder
3. Runs `./run_all_inference.sh` (same as manual `python inference.py` per folder)
4. Reports inference accuracy vs OpenML labels (`pkl/output/inference_report.csv`)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT / 'pkl' / 'scripts'))

from training_common import (
    CANONICAL_INPUT,
    EXTENSIONS,
    deploy_all,
    ensure_canonical_input,
    get_inference_row_indices,
    get_training_data,
    load_feature_columns,
)

PYTHON = sys.executable
FEATURE_COLS = load_feature_columns()
print('ROOT:', ROOT)
print('Extensions:', len(EXTENSIONS), EXTENSIONS)
print('Feature columns:', len(FEATURE_COLS))

## 1. Canonical input dataset

Columns: `id` (serial 1–50), 48 credit features from `schema.json`, `class` (empty).

In [ ]:
ensure_canonical_input()
input_df = pd.read_csv(CANONICAL_INPUT)
print('Path:', CANONICAL_INPUT)
print('Shape:', input_df.shape)
print('Columns (first/last):', list(input_df.columns[:3]), '...', list(input_df.columns[-2:]))
print('Empty class:', input_df['class'].isna().all() or (input_df['class'].astype(str).str.strip() == '').all())
input_df.head()

## 2. Create all model artifacts

Runs `create_model.py` in each extension (trained on OpenML German Credit, 1000 rows, same 48 features).

In [ ]:
create_results = []

for ext in EXTENSIONS:
    ext_dir = ROOT / ext
    print(f'Creating model: {ext}')
    proc = subprocess.run(
        [PYTHON, 'create_model.py'],
        cwd=ext_dir,
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise RuntimeError(f'create_model.py failed for {ext}')

    model_files = sorted((ext_dir / 'model').glob('model.*'))
    model_dirs = [p for p in (ext_dir / 'model').iterdir() if p.is_dir()]
    artifact = (
        model_files[0].name
        if model_files
        else (model_dirs[0].name + '/')
        if model_dirs
        else 'model/'
    )
    create_results.append({'extension': ext, 'status': 'ok', 'artifact': artifact})
    if proc.stdout.strip():
        print(' ', proc.stdout.strip())

pd.DataFrame(create_results)

## 3. Run `run_all_inference.sh`

Deploys `schema.json` + `dataset/input.csv`, then runs `python inference.py` in each extension folder.

In [ ]:
deploy_all()

env = os.environ.copy()
env['PYTHON'] = PYTHON
shell_proc = subprocess.run(
    ['./run_all_inference.sh'],
    cwd=ROOT,
    env=env,
    capture_output=True,
    text=True,
)
print(shell_proc.stdout)
if shell_proc.returncode != 0:
    print(shell_proc.stderr)
    raise RuntimeError('run_all_inference.sh failed')
print('Shell script exit code:', shell_proc.returncode)

## 4. Inference accuracy report

Compares each extension's `output/output.csv` to OpenML labels for the 50 deployment rows.

In [ ]:
_, y, train_idx, test_idx, _ = get_training_data()
inf_indices = get_inference_row_indices()
y_inf = y[inf_indices]
inf_test_mask = np.isin(inf_indices, test_idx)

inference_rows = []
for ext in EXTENSIONS:
    out_df = pd.read_csv(ROOT / ext / 'output' / 'output.csv')
    preds = out_df['target'].to_numpy()
    inference_rows.append({
        'extension': ext,
        'output_rows': len(preds),
        'inference_test_accuracy': round(
            float(accuracy_score(y_inf[inf_test_mask], preds[inf_test_mask])), 4
        )
        if inf_test_mask.any()
        else None,
        'inference_full_accuracy': round(float(accuracy_score(y_inf, preds)), 4),
    })

results_df = pd.DataFrame(inference_rows).sort_values(
    'inference_test_accuracy', ascending=False
)
report_path = ROOT / 'pkl' / 'output' / 'inference_report.csv'
report_path.parent.mkdir(parents=True, exist_ok=True)
results_df.sort_values('extension').to_csv(report_path, index=False)

print('Saved', report_path)
results_df

In [ ]:
results_df['above_80pct_test'] = results_df['inference_test_accuracy'] >= 0.8
results_df